In [ ]:
import os

In [ ]:
from dotenv import load_dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
load_dotenv()

/////INDEXING (DOCUMENT INGESTION)

In [ ]:
"""Fetches the transcript of a YouTube video."""
video_id = os.getenv("YOUTUBE_VIDEO_ID")
if not video_id:
    raise ValueError("YOUTUBE_VIDEO_ID environment variable is missing. Please check your .env file.")
try:
    
     ytt_api = YouTubeTranscriptApi()
     transcript_list = ytt_api.fetch(video_id, languages=["en"])
    
     transcript = " ".join(chunk.text for chunk in transcript_list)
     print(transcript)
    
except TranscriptsDisabled:
    print("No captions available for this video.")

//////INDEXING (TEXT SPLITTING)

In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap=200)
chunks=splitter.create_documents([transcript])

In [ ]:
print(chunks)

In [ ]:
print(chunks[0])

//// EMBEDDING GENERATION & STORING IN VECTORS

In [ ]:
embedding=OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=FAISS.from_documents(chunks,embedding)

In [ ]:
vector_store.docstore._dict

In [ ]:
result = vector_store.get_by_ids(['af34bb98-85f1-4f7c-9f11-df3aa7dc42e0'])
print(result)   

/////RETRIEVAL

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke('What are ai agents?')

/////STEP:3 AUGMENTATION

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

    {context}
    Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question       = "were the principles of making an effective agent discussed? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [ ]:
print(retrieved_docs)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt 

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

////////////STEP 4: GENERATION

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

/////////////////BUILDING A CHAIN

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('What is CTR?') 

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

//////////////EVALUATION METRICS

In [ ]:
import json
import pandas as pd
from datasets import Dataset
from langchain_core.prompts import PromptTemplate
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

print("1. Generating dynamic test questions from the transcript...")

# Prompt for generating Q&A pairs
qa_gen_prompt = PromptTemplate(
    template="""
    You are an expert evaluator. Read the following video transcript and generate 4 factual questions and their correct answers based ONLY on the text.
    Return the output STRICTLY as a JSON format exactly like this, with no extra text or markdown:
    {{
        "qa_pairs": [
            {{"question": "Question 1", "ground_truth": "Answer 1"}},
            {{"question": "Question 2", "ground_truth": "Answer 2"}}
        ]
    }}

    Transcript:
    {transcript}
    """,
    input_variables=["transcript"]
)

# Generate JSON
qa_chain = qa_gen_prompt | llm | parser
raw_qa_output = qa_chain.invoke({"transcript": transcript})

# Parse JSON safely
clean_json = raw_qa_output.replace("```json", "").replace("```", "").strip()
generated_qa = json.loads(clean_json)

# Create variables for evaluation
dynamic_questions = [item["question"] for item in generated_qa["qa_pairs"]]
# FIX 1: Remove the extra brackets so it creates a flat list of strings, not a list of lists
dynamic_ground_truths = [item["ground_truth"] for item in generated_qa["qa_pairs"]]

print("Questions Generated Successfully!")
for q in dynamic_questions:
    print(f"- {q}")

# ---------------------------------------------------------
print("\n2. Running your RAG pipeline to get answers and contexts...")
contexts = []
answers = []

for query in dynamic_questions:
    # Get the generated answer using your existing chain
    response = main_chain.invoke(query)
    answers.append(response)
    
    # Get the raw retrieved documents using your existing retriever
    retrieved_docs = retriever.invoke(query)
    contexts.append([doc.page_content for doc in retrieved_docs])

# Format into a HuggingFace Dataset
# FIX 2: Rename the keys to match Ragas v0.2.0+ requirements
data = {
    "user_input": dynamic_questions,         # Changed from "question"
    "response": answers,                     # Changed from "answer"
    "retrieved_contexts": contexts,          # Changed from "contexts"
    "reference": dynamic_ground_truths       # Changed from "ground_truth"
}
evaluation_dataset = Dataset.from_dict(data)

# ---------------------------------------------------------
print("\n3. Running RAGAS Evaluation (This checks Faithfulness, Relevancy, Precision, and Recall)...")
# Calculate the final scores
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
)

print("\n4. Evaluation Complete! Here is your accuracy report:")
# Convert to a Pandas table and display
df_results = result.to_pandas()
display(df_results)

# Optional: Calculate overall average score
average_score = df_results[['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']].mean().mean()
print(f"\nOverall Pipeline Accuracy Score: {average_score:.2f} out of 1.0")